# # SPATIAL INTELLIGENCE ANALYSIS ON TYPICAL RESIDENTIAL FLOOR PLAN -03

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [35]:
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.33) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [3]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [5]:
objects = Topology.ByOBJPath(r"D:\MACAD\3rd-SM\GML\python\GML.26\Asset\plan.obj")
print(objects)

[<topologic_core.Cluster object at 0x000001D05E226830>, <topologic_core.Cluster object at 0x000001D03EAA3770>]


In [11]:
cluster = Cluster.ByTopologies(objects)
output_path = r"D:\MACAD\3rd-SM\GML\python\GML.26\Asset\resi floor_export_surface.brep"
status = Topology.ExportToBREP(cluster, path=output_path, overwrite=True)
print(f"Export status: {status}")
print(f"BREP file saved to: {output_path}")

Export status: True
BREP file saved to: D:\MACAD\3rd-SM\GML\python\GML.26\Asset\resi floor_export_surface.brep


In [15]:
# Use the BREP generated in the previous step
gallery = Topology.ByBREPPath(output_path)

## 6. Show the geometry

In [9]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

Topology.Show - Error: the input topologies parameter does not contain any valid topology. Returning None.


## 7. Create a grid overlay

In [16]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

if width is None or length is None:
    raise ValueError("Bounding rectangle failed. Make sure gallery is loaded correctly before creating the grid.")

# Set both grid cell sizes to 30% of their previous values (exact scaling)
vertex_grid_step = 28 * 0.3
edge_grid_step = 3 * 0.3

uRange1 = [i * vertex_grid_step for i in range(int(width / vertex_grid_step) + 2)]
vRange1 = [i * vertex_grid_step for i in range(int(length / vertex_grid_step) + 2)]

uRange2 = [i * edge_grid_step for i in range(int(width / edge_grid_step) + 2)]
vRange2 = [i * edge_grid_step for i in range(int(length / edge_grid_step) + 2)]
grid1 = Grid.VerticesByDistances(gallery, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange2, vRange=vRange2)

## 8. Slice the floor plan with the edge grid to create a topologic shell

In [17]:
shell = Topology.Slice(gallery, grid2)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 9. Derive analysis graphs from the shell

In [18]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

## 10. Derive and store the graph vertices and isovist vertices

In [ ]:
g_verts = Graph.Vertices(analysis_graph)
# Use shell-face centroids as viewpoint seeds so isovists start from valid interior points
iso_verts = [Topology.Centroid(f) for f in faces if f]

## 11. Spatial Intelligence through Isovists and Graph Analysis

## Visibility Graph Analysis
### Create Isovists
* Time consuming (about 20 minutes!)
* Will print out errors. Ignore.

In [38]:
# Build isovists by testing each viewpoint against all valid gallery faces
gallery_faces = [f for f in Topology.Faces(gallery) if f]
if not gallery_faces:
    raise ValueError("No valid faces found in gallery for isovist computation.")

isovists = []
valid_iso_verts = []
for v in iso_verts:
    found = None
    for host_face in gallery_faces:
        cand = Face.Isovist(host_face, v)
        if cand:
            found = cand
            break
    if found:
        isovists.append(found)
        valid_iso_verts.append(v)

print(f"Computed {len(isovists)} valid isovists out of {len(iso_verts)} viewpoints.")

Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction. Returning None.
Face.Isovist - Error: The viewpoint is not inside the navigable face after obstacle subtraction.

In [39]:
if not isovists:
    print("No valid isovists to display.")
else:
    Topology.Show(gallery, isovists,
                  faceColorKey="cp_color",
                  faceOpacity=0.6,
                  showEdges=False,
                  showVertices=False,
                  camera=[0,0,6],
                  backgroundColor="black",
                  width=800,
                  height=600,
                  renderer=renderer)

### Compute the visibility of each isovist viewpoint
* Calculate how many other points of the dense grid are within each isovist's face.

In [40]:
new_verts = []
n_list = []

if not isovists:
    print("No valid isovists found. Assigning visibility = 0 to all analysis vertices.")
    for v in g_verts:
        d = Topology.Dictionary(v)
        d = Dictionary.SetValueAtKey(d, "visibility", 0)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)
    n_list = [0]
else:
    for i, iso in enumerate(isovists):
        v = valid_iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)

### Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [41]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

### Derive the colour of each vertex based on the interpolated value

In [42]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    if minValue == maxValue:
        color = Color.AnyToHex(Color.ByValueInRange(0.5, minValue=0, maxValue=1, colorScale="thermal"))
    else:
        color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

### Transfer the information from the graph vertices to the faces of the original shell

In [43]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [44]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)